# Demo 02 — Pipeline Tour: How We Build Sentence-Level Citations

*For the customer exec, PM, or policy auditor who just watched Demo 01 and asked
"...okay, but how?"*

---

## What this notebook shows

A narrated walkthrough of the **six-step pipeline** that turns a folder of IRS
PDFs into answers where every sentence is anchored to a source sentence:

1. **Ingestion** — OCR + layout extraction (Azure Document Intelligence)
2. **Chunking** — sentence-aware splits with stable sentence IDs
3. **Indexing** — dual-layout Azure AI Search (chunks + sentences)
4. **Retrieval** — hybrid BM25 + vector, union across both layouts
5. **Generation** — two strategies for attaching citations
6. **Verification** — what the auditor actually clicks on

Each step is illustrated with **real artifacts** from a cached pipeline run.

## Prerequisites

- You've read [`demo_01_the_problem.ipynb`](demo_01_the_problem.ipynb) — that
  notebook motivates **why** sentence-level citations matter. This notebook
  answers **how** we produce them.

## 🔌 Zero live calls, runs anywhere

Everything in this notebook reads from `data/notebook_cache/`:

- **No Azure** (no Document Intelligence, no AI Search, no OpenAI)
- **No LLM calls**
- **No `.env` required**
- **No network access**

If the cache files are present, this notebook runs on a laptop with no config.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

CACHE = Path("../data/notebook_cache")
DOC_ID = "irs_pub_583"

pd.set_option("display.max_colwidth", 120)


## Step 0 — The corpus

The prototype indexes a small set of **IRS publications** (the kind of guidance
a tax preparer or compliance reviewer works with daily). For this demo we'll
follow one document through the entire pipeline: **IRS Publication 583 —
*Starting a Business and Keeping Records***.

The cached `meta.json` captures everything we know about the source document
*before* we touch it: page count, file hash, when it was parsed, which Document
Intelligence model produced the layout.


In [ ]:
meta = json.loads((CACHE / "parsed" / DOC_ID / "meta.json").read_text())

pd.DataFrame([{
    "document_id": meta["document_id"],
    "filename": meta["filename"],
    "pages": meta["page_count"],
    "sha256 (first 12)": meta["sha256"][:12] + "…",
    "parsed_at": meta["parsed_at"],
    "di_model": meta["di_model_id"],
    "di_api_version": meta["di_api_version"],
}]).T.rename(columns={0: "value"})


## Step 1 — Ingestion: OCR + layout extraction

**What happens:** the PDF is sent to **Azure Document Intelligence** (`prebuilt-layout`),
which returns a structured JSON describing every page, paragraph, table, and
bounding box. We also persist a markdown rendering for human-readable debugging.

**Why it matters:** IRS PDFs have multi-column layouts, tables, and OCR-only
scanned sections. Naïve `pdftotext` loses structure; DI preserves it.

**Cost shape:** this step is **expensive and one-time per document**. Once
`layout.json` is cached, every downstream step is free to re-run.


In [ ]:
parsed_dir = CACHE / "parsed" / DOC_ID
layout_path = parsed_dir / "layout.json"
md_path = parsed_dir / "document.md"

# Show file sizes so the reader sees the scale of the DI output
sizes = pd.DataFrame([
    {"artifact": "layout.json (raw DI output)",
     "size": f"{layout_path.stat().st_size / 1_000_000:.1f} MB",
     "purpose": "Structured layout: pages, paragraphs, tables, bboxes"},
    {"artifact": "document.md (human-readable)",
     "size": f"{md_path.stat().st_size / 1_000:.0f} KB",
     "purpose": "Flattened markdown for inspection & chunking input"},
])
display(sizes)

# Preview the first 800 chars of the markdown rendering
preview = md_path.read_text()[:800]
display(Markdown("**First 800 characters of `document.md`:**\n\n```\n" + preview + "\n```"))


## Step 2 — Chunking: sentence-aware splits

**What happens:** we split each document into overlapping **chunks** (a few
hundred tokens each), and inside each chunk we break the text into **sentences**.
Every sentence gets a stable identifier — the `sentence_id` (we call it a
**sid**) — of the form `{document_id}-s{NNNNN}`.

**Why it matters:** the sid is the atom of citation. Everything downstream —
the index, the retriever, the citation alignment math — pivots on sids. If a
sentence moves, its sid follows it, so citations stay correct across reindexes.

**Shape of the cache:** `data/notebook_cache/chunks/irs_pub_583.jsonl` —
one chunk per line, each chunk carries its nested sentences with char offsets
and page numbers.


In [ ]:
chunks_path = CACHE / "chunks" / f"{DOC_ID}.jsonl"
chunks = [json.loads(line) for line in chunks_path.read_text().splitlines()]

print(f"Total chunks for {DOC_ID}: {len(chunks):,}")
print(f"Total sentences       : {sum(len(c['sentences']) for c in chunks):,}")

# Pick the first two substantive chunks (skip the tiny title-page chunk)
sample_chunks = [c for c in chunks if len(c["sentences"]) >= 2][:2]

rows = []
for ch in sample_chunks:
    rows.append({
        "chunk_id": ch["chunk_id"],
        "page": ch["page"],
        "n_sentences": len(ch["sentences"]),
        "token_count": ch["token_count"],
        "section_path": " > ".join(ch["section_path"]) or "—",
        "text (first 140 chars)": ch["text"][:140] + ("…" if len(ch["text"]) > 140 else ""),
    })
display(pd.DataFrame(rows))


In [ ]:
# Drill into one chunk to show the nested sentence array — this is what makes
# sentence-level citation possible.
ch = sample_chunks[1]
sent_rows = [{
    "sentence_id": s["sentence_id"],
    "page": s["page"],
    "char_start": s["char_start"],
    "char_end": s["char_end"],
    "text": s["text"][:110] + ("…" if len(s["text"]) > 110 else ""),
} for s in ch["sentences"][:6]]
display(Markdown(f"**Sentences inside `{ch['chunk_id']}` (first 6 of {len(ch['sentences'])}):**"))
display(pd.DataFrame(sent_rows))


## Step 3 — Indexing: dual-layout Azure AI Search

**What happens:** we push the chunks into **two Azure AI Search indexes**
simultaneously. Same content, different primary unit:

| Index          | Primary doc | What it's good at                                  |
|----------------|-------------|----------------------------------------------------|
| **Layout X**   | Chunk       | Broad context, long-range reasoning, table-aware   |
| **Layout Y**   | Sentence    | Precise hits, exact-quote retrieval, tight recall  |

Both indexes carry **BM25 text fields** *and* **vector fields** (embeddings),
so each can be searched hybrid.

**Why two?** Neither unit alone is enough:

```
              ┌────────────────────┐
  query  ───▶ │  Layout X (chunks) │ ──▶ hits_X ─┐
              └────────────────────┘              │
                                                  ├──▶ union ──▶ retrieval result
              ┌────────────────────┐              │
  query  ───▶ │ Layout Y (sentences)│ ──▶ hits_Y ─┘
              └────────────────────┘
```

- Chunks-only retrieval returns too much text for a citation to be precise.
- Sentences-only retrieval loses the surrounding context the LLM needs.
- **Union** gives the LLM long-context chunks *and* a menu of precise sentences
  — the best of both.

No code in this step: this is a pure infrastructure step. The artifact is the
retrieval snapshot you'll see next.


## Step 4 — Retrieval: hybrid, dual-mode union

**What happens at query time:**

1. Embed the question (one embedding call).
2. Run hybrid (BM25 + vector) search against **Layout X** → top-K chunks.
3. Run hybrid search against **Layout Y** → top-K sentences.
4. **Union** the results: keep the chunks, and for any sentence hit whose
   parent chunk isn't already in the set, pull the parent chunk in too
   (`parent_chunks_added` in the snapshot tracks this).
5. Return a packaged `RetrievalResult` — chunks + a candidate sentence menu.

**Cached artifact:** `data/notebook_cache/retrieval/snapshot.json` — the full
retrieval result for **the same question Demo 01 used**, so everything lines up
across the demos.


In [ ]:
snapshot = json.loads((CACHE / "retrieval" / "snapshot.json").read_text())

display(Markdown(
    f"**Query:** _{snapshot['query']}_  \n"
    f"**Mode:** `{snapshot['mode']}`  \n"
    f"**Latency:** {snapshot['latency_ms']:.0f} ms  \n"
    f"**Chunk-index hits:** {snapshot['chunk_search_hits']}  "
    f"**Sentence-index hits:** {snapshot['sentence_search_hits']}  "
    f"**Parent chunks pulled in:** {snapshot['parent_chunks_added']}"
))

# Top-5 sentence candidates — the precise menu the generator will cite from
cand_rows = []
for c in snapshot["sentence_candidates"][:5]:
    cand_rows.append({
        "sid": c["sentence_id"],
        "source": c["document_id"],
        "page": c.get("page"),
        "score": round(c.get("score", 0.0), 3),
        "text_snippet": (c["text"][:130] + "…") if len(c["text"]) > 130 else c["text"],
    })
display(Markdown("**Top-5 sentence candidates (Layout Y hits):**"))
display(pd.DataFrame(cand_rows))


## Step 5 — Generation: two strategies

Once we have the retrieval result, we hand **chunks + sentence menu + question**
to the LLM. There are two ways to get sentence-level citations out the other
side:

### Strategy A — `inline_prompted`
The prompt instructs the model to emit `[sid]` markers inline **as it writes**
the answer. The model is told exactly which sids it's allowed to cite (the
menu from Layout Y).

- ✅ Simple: one LLM call, citations come out fused with the answer.
- ⚠️ The model can hallucinate sids or attach them to the wrong sentence.

### Strategy B — `post_gen_alignment`
The model writes a clean, un-cited answer. Then a **separate deterministic
alignment step** (embedding similarity + lexical overlap) matches each answer
sentence to the best supporting sids from the candidate menu.

- ✅ The model doesn't need to juggle citation syntax while reasoning.
- ⚠️ Needs a second pass; cost is ~1.5× the inline strategy.

**Which is better?** Different tradeoffs — precision vs. stability vs. coverage.
That's what Demo 03 measures. For now, let's look at a real `inline_prompted`
answer from the cache.


In [ ]:
eval_items = [json.loads(line) for line in (CACHE / "eval" / "items.jsonl").open()]

# Use the same question as Demo 01 — it's the one the retrieval snapshot is for.
item = next(it for it in eval_items if it["question"] == snapshot["query"])

display(Markdown(
    f"**Question:** {item['question']}  \n"
    f"**Strategy:** `{item['strategy']}`  \n"
    f"**Answer sentences:** {item['n_answer_sentences']}  "
    f"**Predicted citations:** {item['n_pred_ids']}  "
    f"**Latency:** {item['latency_ms']:.0f} ms"
))

display(Markdown("**Raw answer (with inline sid citations):**\n\n> " +
                 item["answer_text"]))

display(Markdown("**Predicted citation sids:** `" +
                 "`, `".join(item["pred_citation_ids"]) + "`"))


## Step 6 — Verification: what the auditor sees

Every sentence in the answer carries one or more sids. A sid is a **direct
handle** back into the source corpus — no guessing, no re-reading paragraphs to
find the supporting clause.

The table below is the auditor's view: for each cited sid, we show the exact
source sentence, which document it came from, and which page it's on. This is
the artifact that makes "click the sid, jump to the source" work in the app.


In [ ]:
# Build a sid -> (text, document, page) lookup from the retrieval snapshot.
# The snapshot carries both the chunks (with nested sentences) and the
# sentence-candidate menu — union them.
sid_lookup = {}
for ch in snapshot["chunks"]:
    for s in ch.get("sentences") or []:
        sid_lookup[s["sentence_id"]] = {
            "text": s["text"],
            "document_id": ch["document_id"],
            "page": s.get("page", ch.get("page")),
        }
for s in snapshot["sentence_candidates"]:
    sid_lookup.setdefault(s["sentence_id"], {
        "text": s["text"],
        "document_id": s["document_id"],
        "page": s.get("page"),
    })

rows = []
for sid in item["pred_citation_ids"]:
    info = sid_lookup.get(sid)
    rows.append({
        "sid": sid,
        "source": info["document_id"] if info else "—",
        "page": info["page"] if info else "—",
        "source sentence": (info["text"] if info else "(not in cached snapshot)")[:180]
        + ("…" if info and len(info["text"]) > 180 else ""),
    })

display(Markdown("**sid → source-sentence map for the cached answer:**"))
display(pd.DataFrame(rows))


## Recap

Six steps, zero hand-waving:

1. **Ingest** the PDF → structured layout JSON (one-time, cached).
2. **Chunk** into overlapping windows, assign stable sids to every sentence.
3. **Index** twice — chunks for context, sentences for precision.
4. **Retrieve** with hybrid search, union both layouts at query time.
5. **Generate** with either inline sid prompting or post-gen alignment.
6. **Verify** — every answer sentence maps back to a real source sentence.

No sentence in the answer is unanchored. No citation points to "page 49" and
leaves the reviewer to hunt. That's the whole point.

## What's next

- **[`demo_03_metrics_that_matter.ipynb`](demo_03_metrics_that_matter.ipynb)** —
  "sounds plausible, but does it actually work?" We walk through the metrics we
  track (precision, recall, coverage, faithfulness, stability) and show what
  the two generation strategies look like head-to-head on a full eval run.
